In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!",
 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-5c55-6212-8002-18a832f31064'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-10T13:28:04.329477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-42fc-6e9a-8001-c2081882998f'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-5c55-6212-8002-18a832f31064'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-10T13:28:04.329477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-42fc-6e9a-8001-c2081882998f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he 

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti blush? Because it saw the pasta sauce!',
 'explanation': 'This joke is a play on words that uses the idea of spaghetti blushing, which is obviously not something that can actually happen. The punchline is a pun, as the spaghetti sees the pasta sauce, which causes it to blush. The humor comes from the absurdity of a strand of spaghetti blushing and the unexpected connection between the spaghetti and the pasta sauce.'}

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-5c55-6212-8002-18a832f31064'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-10T13:28:04.329477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-42fc-6e9a-8001-c2081882998f'}}, tasks=(), interrupts=())

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-5c55-6212-8002-18a832f31064'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-10T13:28:04.329477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-42fc-6e9a-8001-c2081882998f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he 

### Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [15]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

EmptyInputError: Received no input for __start__

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him too much "dough-ma" or stress. It\'s a light-hearted and punny way to make a joke about the challenges of making pizza dough.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-5c55-6212-8002-18a832f31064'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-10T13:28:04.329477+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d03-42fc-6e9a-8001-c2081882998f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he 

#### Updating State

In [1]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

NameError: name 'workflow' is not defined

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d12-5b82-6b2a-8000-db8e9c49375c'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-06-10T13:34:46.896498+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='220237f4-ccdc-b03f-4e3a-156fcdd714d0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him

In [19]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

EmptyInputError: Received no input for __start__

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f164d12-5b82-6b2a-8000-db8e9c49375c'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-06-10T13:34:46.896498+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='220237f4-ccdc-b03f-4e3a-156fcdd714d0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't handle all the dough-ma!", 'explanation': 'This joke is a play on words, as "dough-ma" sounds like "drama." The joke is suggesting that the pizza maker went to therapy because he was struggling to handle all the dough (the main ingredient of pizza) and it was causing him

### Fault Tolerance

In [21]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [22]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [23]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [24]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))